# 05 - Hopsworks Feature Store

## Objective

Register the engineered Lahore air quality features in the Hopsworks Feature Store.

### Input

`data/processed/lahore/lahore_features_hourly.csv`

### Output

Hopsworks Feature Group:

`lahore_air_quality_features`

### Feature Store

Hopsworks

### Primary Key

`datetime`

### Event Time

`datetime`

In [5]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
import hopsworks

print("Imports successful.")

Imports successful.


In [6]:
# Load environment variables from .env
load_dotenv()

HOPSWORKS_API_KEY = os.getenv("HOPSWORKS_API_KEY")

if not HOPSWORKS_API_KEY:
    raise ValueError(
        "HOPSWORKS_API_KEY not found. "
        "Make sure it is defined in the .env file."
    )

print("Hopsworks API key loaded successfully.")

Hopsworks API key loaded successfully.


In [7]:
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd()

# Feature dataset
FEATURE_DATA_PATH = (
    PROJECT_ROOT
    / "../data"
    / "processed"
    / "lahore"
    / "lahore_features_hourly.csv"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nFeature dataset:")
print(FEATURE_DATA_PATH)

print("\nExists:", FEATURE_DATA_PATH.exists())

Project root:
d:\talha\Pearls_AQI_Predictor\notebooks

Feature dataset:
d:\talha\Pearls_AQI_Predictor\notebooks\..\data\processed\lahore\lahore_features_hourly.csv

Exists: True


In [8]:
lahore_features = pd.read_csv(
    FEATURE_DATA_PATH,
    parse_dates=["datetime"]
)

print("Dataset loaded successfully.")
print("Shape:", lahore_features.shape)

display(lahore_features.head())

Dataset loaded successfully.
Shape: (22608, 30)


,datetime,temperature_2m,relative_humidity_2m,surface_pressure,precipitation,cloud_cover,wind_speed_10m,wind_direction_10m,pm2_5,pm10,...,hour_sin,hour_cos,month_sin,month_cos,aqi_change_rate,aqi_lag_1h,aqi_lag_24h,aqi_rolling_mean_6h,aqi_rolling_std_6h,pm25_rolling_mean_6h
0,2024-01-02 00:00:00,6.15,99.312260,992.20780,0.0,34.0,5.692099,214.69522,253.3,363.2,...,0.000000,1.000000,0.5,0.866025,-0.34585,265.12085,272.10837,265.722912,0.729160,236.816667
1,2024-01-02 01:00:00,5.40,99.653530,991.74695,0.0,100.0,6.369050,227.29063,248.7,357.6,...,0.258819,0.965926,0.5,0.866025,-0.31254,264.77500,272.72916,265.349295,0.699532,244.500000
2,2024-01-02 02:00:00,5.00,99.305890,991.61150,0.0,100.0,4.104631,254.74483,234.2,336.2,...,0.500000,0.866025,0.5,0.866025,-0.50409,264.46246,273.39163,264.956248,0.709747,246.350000
3,2024-01-02 03:00:00,5.15,100.000000,991.43110,0.0,100.0,5.937272,284.03625,218.9,314.8,...,0.707107,0.707107,0.5,0.866025,-0.21249,263.95837,273.42917,264.595148,0.676307,243.166667
4,2024-01-02 04:00:00,5.20,99.652985,991.14380,0.0,100.0,3.319036,310.60123,198.4,285.9,...,0.866025,0.500000,0.5,0.866025,0.42917,263.74588,273.07916,264.372935,0.516385,234.633333


In [9]:
print("=" * 60)
print("FEATURE DATASET VALIDATION")
print("=" * 60)

print("Shape:", lahore_features.shape)

print(
    "Start:",
    lahore_features["datetime"].min()
)

print(
    "End:",
    lahore_features["datetime"].max()
)

print(
    "Duplicate timestamps:",
    lahore_features["datetime"].duplicated().sum()
)

print(
    "Missing values:",
    lahore_features.isnull().sum().sum()
)

print(
    "Non-hourly intervals:",
    (
        lahore_features["datetime"]
        .diff()
        .dropna()
        != pd.Timedelta(hours=1)
    ).sum()
)

print("=" * 60)

FEATURE DATASET VALIDATION
Shape: (22608, 30)
Start: 2024-01-02 00:00:00
End: 2026-07-31 23:00:00
Duplicate timestamps: 0
Missing values: 0
Non-hourly intervals: 0


In [10]:
lahore_features.dtypes

datetime                datetime64[ns]
temperature_2m                 float64
relative_humidity_2m           float64
surface_pressure               float64
precipitation                  float64
cloud_cover                    float64
wind_speed_10m                 float64
wind_direction_10m             float64
pm2_5                          float64
pm10                           float64
carbon_monoxide                float64
nitrogen_dioxide               float64
sulphur_dioxide                float64
ozone                          float64
us_aqi                         float64
hour                             int64
day                              int64
month                            int64
day_of_week                      int64
is_weekend                       int64
hour_sin                       float64
hour_cos                       float64
month_sin                      float64
month_cos                      float64
aqi_change_rate                float64
aqi_lag_1h               

In [11]:
import pyarrow
print(pyarrow.__version__)

25.0.1


In [12]:
import pyarrow._flight

In [13]:
import hopsworks

print("Hopsworks imported successfully")
print("Version:", hopsworks.__version__)

Hopsworks imported successfully
Version: 5.0.4


In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

HOPSWORKS_API_KEY = os.getenv("HOPSWORKS_API_KEY")

print("API key loaded:", HOPSWORKS_API_KEY is not None)

API key loaded: True


In [15]:
import hopsworks

project = hopsworks.login(
    api_key_value=HOPSWORKS_API_KEY,
    cert_folder=r"D:\talha\Pearls_AQI_Predictor\hopsworks-certs"
)

print("Connected to Hopsworks successfully.")
print("Project:", project.name)

2026-08-16 22:42:58,333 INFO: Initializing external client
2026-08-16 22:42:58,336 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-08-16 22:43:03,870 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42187
Connected to Hopsworks successfully.
Project: internship10P


In [16]:
feature_store = project.get_feature_store()

print("Feature Store connected successfully.")

Feature Store connected successfully.


In [17]:
primary_key=["datetime"]
event_time="datetime"

In [18]:
import deltalake

print("Delta Lake:", deltalake.__version__)

Delta Lake: 1.6.2


In [19]:
feature_group = feature_store.get_or_create_feature_group(
    name="lahore_air_quality_features",
    version=1,
    description=(
        "Hourly weather, pollutant, AQI, temporal, "
        "lag, and rolling features for Lahore air quality prediction."
    ),
    primary_key=["datetime"],
    event_time="datetime",
    online_enabled=True
)

print("Feature Group ready.")
print("Name:", feature_group.name)
print("Version:", feature_group.version)

Feature Group ready.
Name: lahore_air_quality_features
Version: 1


In [ ]:
print("Uploading feature data to Hopsworks...")

job, validation_report = feature_group.insert(
    lahore_features,
    wait=True
)

print("Feature data inserted successfully.")

In [ ]:
print("Reading data back from Hopsworks...")

hopsworks_df = feature_group.read()

print("Shape:", hopsworks_df.shape)

display(hopsworks_df.head())

In [ ]:
print("=" * 60)
print("HOPSWORKS FEATURE STORE VERIFICATION")
print("=" * 60)

print("Rows:", len(hopsworks_df))
print("Columns:", len(hopsworks_df.columns))

print(
    "Minimum datetime:",
    hopsworks_df["datetime"].min()
)

print(
    "Maximum datetime:",
    hopsworks_df["datetime"].max()
)

print(
    "Missing values:",
    hopsworks_df.isnull().sum().sum()
)

print("=" * 60)